In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"

spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
%sql
SHOW SCHEMAS IN marathos;

In [0]:
df = spark.sql("FROM marathos.bronze.raw_supply_chain")
df.display()

### Change alla column names to snake case for consistency

In [0]:
import re

# Ex. Event name -> event_name
def to_snake_case(name):
    return re.sub(r"[\s]+","_",name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case("Event name")

In [0]:
df_cleaned_columns = rename_columns_to_snake_case(df)
df_cleaned_columns.display()

In [0]:
from pyspark.sql.functions import col, when, regexp_extract, round as spark_round

df_silver = ( 
    df_cleaned_columns
    .withColumn(
        "athlete_year_of_birth",
        when(
            col("athlete_year_of_birth").cast("int").between(1900, 2010),
            col("athlete_year_of_birth").cast("int")
        ).otherwise(None)
    )
    .withColumn(
        "performance_seconds",
        when(
            col("athlete_performance").rlike(r"^\d+:\d{2}:\d{2}\s*h$"),
            regexp_extract(col("athlete_performance"), r"^(\d+)", 1).cast("double") * 3600 +
            regexp_extract(col("athlete_performance"), r":(\d{2}):", 1).cast("double")* 60 +
            regexp_extract(col("athlete_performance"), r":(\d{2})\s*h", 1).cast("double")
        ).otherwise(None)
    )
    .withColumn(
        "performance_km",
        when(
            col("athlete_performance").rlike(r"^\d+\.?\d*\s*km$"),
            regexp_extract(col("athlete_performance"), r"^(\d+\.?\d*)", 1).cast("double")
        ).otherwise(None)
    )
    .withColumn(
        "event_hours",
        when(
            col("event_distance/length").rlike(r"^\d+h$"),
            regexp_extract(col("event_distance/length"), r"^(\d+)", 1).cast("double")
        ).otherwise(None)
    )
    .withColumn(
        "event_distance_km",
        when(
            col("event_distance/length").rlike(r"^\d+\.?\d*\s*km$"),
            regexp_extract(col("event_distance/length"), r"^(\d+\.?\d*)", 1).cast("double")
        )
        .when(
            col("event_distance/length").rlike(r"^\d+\.?\d*mi$"),
            regexp_extract(col("event_distance/length"), (r"^(\d+\.?\d*)"), 1).cast("double") * 1.60934
        ).otherwise(None)
    )
    # fix athlete average speed by using the following formula:
    # athlete_average_speed = performance_km / (performance_seconds / 3600)
    .withColumn(
        "athlete_average_speed",
        when(
            col("event_hours").isNotNull() & col("performance_km").isNotNull() & (col("event_hours") > 0),
            spark_round(col("performance_km") / col("event_hours"), 3)
        )
        .when(
            col("event_distance_km").isNotNull() & col("performance_seconds").isNotNull() & (col("performance_seconds") > 0),
            spark_round(col("event_distance_km") / (col("performance_seconds") / 3600), 3)
        )
        .otherwise(None)
    )
    .filter(~col("athlete_performance").rlike(r"^\d+d\s+"))
    .filter(~col("event_distance/length").rlike(r"(?i)\bd\b"))
    .filter(col("athlete_average_speed").between(0.5, 20))
.filter(col("athlete_performance").isNotNull())
)

df_silver.printSchema()

In [0]:
df_silver.filter(col("athlete_average_speed").isNull()).count()

In [0]:
df_silver.select("athlete_average_speed").describe().display()

In [0]:
df_silver.filter(col("athlete_average_speed").isNull()).filter(
    col("event_distance/length").rlike(r"^\d+\.?\d*\s*km$")
).count()